In [ ]:
import polars as pl
from pathlib import Path
import shutil

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
DRAW_DIR = BASE / "brms_draws"  # only used to verify the seed 1 rebuild
REFIT_DIR_NAME = "brms_refits"

PRIMARY_SEED = 1  # chosen a priori, before any results
TOP_QUANTILE = 0.99  # drop_top1pct keeps igr_len <= this quantile

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]
polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opp",
    "-+": "opp",
}
polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "conv",
    "-+": "div",
}
allow_polarity = set(polarity_3bin.keys())
trna_labels = {
    "TA",
    "TASX",
    "ASX",
    "TC",
    "TD",
    "TE",
    "TER",
    "TF",
    "TFM",
    "TG",
    "TGLX",
    "TH",
    "TI",
    "TK",
    "TL1|2",
    "TM",
    "TN",
    "TP",
    "TQ",
    "TR",
    "TRMR",
    "TRNX",
    "TS1|2",
    "TSEC",
    "TSUP",
    "TT",
    "TU",
    "TV",
    "TW",
    "TX",
    "TXLE",
    "XLE",
    "TY",
}
rrna_labels = {
    "RNR12",
    "RNR15",
    "RNR16",
    "RNR18",
    "RNR21",
    "RNR23",
    "RNR26",
    "RNR28",
    "RNR4",
    "RNR4.5",
    "RNR5",
    "RNR9",
    "RNRL",
    "RNRS",
    "RRNA",
    "RRN3",
    "RRN7",
    "RRNF",
    "RRF",
}
intronic_labels = {
    "GIY-YIG",
    "I-END",
    "I-MAT",
    "I-RT",
    "LAGLIDADG",
    "MAT",
    "NAT",
    "RT",
    "LAG-END",
    "MATK",
    "MATR",
    "END-LAG",
}

# Output columns for brms_input.tsv (same as the draws)
brms_cols = [
    "AN",
    "ID",
    "polarity_3bin",
    "igr_len",
    "flanking_types",
    "ncbi_taxid",
    "log10_igr_len",
    "genome_length",
    "log10_genome_length",
    "log10_geom_mean_flanking_gene_len",
    "log10_min_flanking_gene_len",
    "log10_max_flanking_gene_len",
    "sup_len",
    "sdw_len",
    "ncbi_name",
]

In [ ]:
def feature_type(label_col):
    return (
        pl.when(pl.col(label_col).is_in(trna_labels))
        .then(pl.lit("trna"))
        .when(pl.col(label_col).is_in(rrna_labels))
        .then(pl.lit("rrna"))
        .otherwise(pl.lit("gene"))
    )


def build_features(igr, tsv, g, tag=""):
    igr = igr.join(tsv, on="AN", how="inner")

    for side, col in (("sup", "source_up"), ("sdw", "source_dw")):
        igr = (
            igr.with_columns(
                pl.col(col)
                .str.split_exact("|", 5)
                .struct.rename_fields(
                    [
                        f"{side}_an",
                        f"{side}_type",
                        f"{side}_start",
                        f"{side}_end",
                        f"{side}_polarity",
                        f"{side}_gff_id",
                    ]
                )
                .alias("parts")
            )
            .unnest("parts")
            .with_columns(
                pl.col(f"{side}_start").cast(pl.Int64),
                pl.col(f"{side}_end").cast(pl.Int64),
            )
            .drop([f"{side}_an", f"{side}_type", f"{side}_gff_id"])
        )

    before = igr.height
    igr = igr.filter(pl.col("Polarity").is_in(allow_polarity))
    print(f"[{tag}] {g} - {before - igr.height:,} IGRs dropped due to polarity filter")

    igr = igr.with_columns(
        pl.col("Polarity").replace_strict(polarity_3bin).alias("polarity_3bin"),
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin"),
    )
    igr = igr.with_columns(
        (pl.col("sup_end") - pl.col("sup_start") + 1).alias("sup_len"),
        (pl.col("sdw_end") - pl.col("sdw_start") + 1).alias("sdw_len"),
    )
    igr = igr.with_columns(
        pl.col("sup_len").fill_null(pl.col("sdw_len")).alias("sup_len"),
        pl.col("sdw_len").fill_null(pl.col("sup_len")).alias("sdw_len"),
    )
    igr = igr.with_columns(
        ((pl.col("sup_len") + pl.col("sdw_len")) / 2).alias("mean_flanking_gene_len"),
        ((pl.col("sup_len").log10() + pl.col("sdw_len").log10()) / 2).alias(
            "log10_geom_mean_flanking_gene_len"
        ),
        pl.min_horizontal("sup_len", "sdw_len")
        .log10()
        .alias("log10_min_flanking_gene_len"),
        pl.max_horizontal("sup_len", "sdw_len")
        .log10()
        .alias("log10_max_flanking_gene_len"),
    )
    igr = igr.with_columns(
        pl.col("Length").log10().alias("log10_igr_len"),
        pl.col("genome_length").log10().alias("log10_genome_length"),
    )
    igr = igr.rename({"Length": "igr_len"})

    igr = igr.with_columns(
        pl.col("UP").str.replace_all(r"^MIT-|^PLT-", "").alias("label_up"),
        pl.col("DOWN").str.replace_all(r"^MIT-|^PLT-", "").alias("label_dw"),
    )

    igr = igr.with_columns(
        feature_type("label_up").alias("type_up"),
        feature_type("label_dw").alias("type_dw"),
    )
    igr = igr.with_columns(
        pl.when(pl.col("type_up") <= pl.col("type_dw"))
        .then(pl.col("type_up") + "-" + pl.col("type_dw"))
        .otherwise(pl.col("type_dw") + "-" + pl.col("type_up"))
        .alias("flanking_types")
    )
    print(f"[{tag}] {g} {igr['AN'].n_unique():>7,} genomes  {igr.height:>10,} IGRs")
    return igr

In [ ]:
all_data = {}
floored_data = {}
for g in groups:
    gdir = BASE / g
    igr = pl.read_csv(gdir / f"{g}_summary_igr.tsv", separator="\t")
    tsv = (
        pl.read_csv(gdir / f"{g}.tsv", separator="\t")
        .select(["AN", "ncbi_taxid", "Genome_length", "ncbi_name"])
        .rename({"Genome_length": "genome_length"})
    )
    all_data[g] = build_features(
        igr.filter(pl.col("Length") > 0), tsv, g, tag="no-zero"
    )
    igr_floored = igr.with_columns(
        (pl.col("Length") == 0).alias("floored"),
        pl.col("Length").clip(lower_bound=1),
    )
    floored_data[g] = build_features(igr_floored, tsv, g, tag="zero-floor")

In [ ]:
def draw_genus_representatives(g, seed):
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t").select(
        ["AN", "ncbi_genus"]
    )
    return (
        tsv.sample(fraction=1.0, shuffle=True, seed=seed)
        .group_by("ncbi_genus", maintain_order=True)
        .head(1)
    )


print(f"polars {pl.__version__}\n")
primary = {}
for g in groups:
    pick = draw_genus_representatives(g, PRIMARY_SEED)
    ans = pick["AN"].to_list()
    primary[g] = all_data[g].filter(pl.col("AN").is_in(ans))

    ref_dir = DRAW_DIR / g / f"seed_{PRIMARY_SEED}"
    ref_reps = ref_dir / "genus_representatives.tsv"
    ref_input = ref_dir / "brms_input.tsv"
    status = "no reference found, not verified"
    if ref_reps.exists():
        ref_ans = set(pl.read_csv(ref_reps, separator="\t")["AN"])
        if ref_ans != set(ans):
            raise ValueError(
                f"{g}: rebuilt seed {PRIMARY_SEED} representatives differ from "
                f"{ref_reps} ({len(ref_ans ^ set(ans))} genomes differ)"
            )
        status = "representatives match"
        if ref_input.exists():
            ref_ids = pl.read_csv(ref_input, separator="\t", columns=["AN", "ID"])
            new_ids = primary[g].select(["AN", "ID"])
            if (
                ref_ids.height != new_ids.height
                or ref_ids.join(new_ids, on=["AN", "ID"], how="anti").height
            ):
                raise ValueError(f"{g}: rebuilt IGRs differ from {ref_input}")
            status += ", IGRs match"
    print(f"{g:16s} {len(ans):>6,} genomes  {primary[g].height:>9,} IGRs  ({status})")

In [ ]:
def drop_longest(df):
    return (
        df.sort(["AN", "igr_len", "ID"], descending=[False, True, False])
        .with_columns(pl.int_range(pl.len()).over("AN").alias("_rank"))
        .filter(pl.col("_rank") > 0)
        .drop("_rank")
    )


def drop_top1pct(df):
    q = df["igr_len"].quantile(TOP_QUANTILE, interpolation="linear")
    return df.filter(pl.col("igr_len") <= q), q


def gene_gene_only(df):
    return df.filter(pl.col("flanking_types") == "gene-gene")


REFITS = ["drop_longest", "drop_top1pct", "gene_gene_only", "floor_1bp"]
numeric_required = [
    "log10_igr_len",
    "log10_geom_mean_flanking_gene_len",
    "log10_min_flanking_gene_len",
    "log10_max_flanking_gene_len",
]
categorical_required = ["ncbi_taxid", "polarity_3bin", "flanking_types"]

In [ ]:
def polarity_counts(df):
    c = dict(df.group_by("polarity_3bin").len().iter_rows())
    return {k: c.get(k, 0) for k in ("same", "conv", "div")}


def write_refit(g, run, df):
    out = BASE / g / REFIT_DIR_NAME / run
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy(BASE / g / "tree.nwk", out / "tree.nwk")

    cols = brms_cols + (["floored"] if run == "floor_1bp" else [])
    df = df.select(cols)
    for col in numeric_required:
        bad = df.filter(
            pl.col(col).is_null() | pl.col(col).is_nan() | pl.col(col).is_infinite()
        )
        if bad.height:
            print(f"  WARNING [{g}/{run}] {col}: {bad.height} problematic rows")
    for col in categorical_required:
        bad = df.filter(pl.col(col).is_null())
        if bad.height:
            print(f"  WARNING [{g}/{run}] {col}: {bad.height} null rows")

    df = df.with_columns(pl.col(pl.Float32, pl.Float64).round(4))
    df.write_csv(out / "brms_input.tsv", separator="\t")
    return out

In [ ]:
for g in groups:
    base_df = primary[g].with_row_index("_order")
    base_counts = polarity_counts(base_df)
    base_ans = set(base_df["AN"].unique())
    print(
        f"## {g}  (seed {PRIMARY_SEED}: {base_df.height:,} IGRs, {len(base_ans):,} genomes; "
        f"same {base_counts['same']:,} | conv {base_counts['conv']:,} | div {base_counts['div']:,})"
    )

    for run in REFITS:
        extra = ""
        if run == "drop_longest":
            df = drop_longest(base_df)

        elif run == "drop_top1pct":
            df, q = drop_top1pct(base_df)
            extra = f"  q{TOP_QUANTILE * 100:g} = {q:,.1f} bp"

        elif run == "floor_1bp":
            df = floored_data[g].filter(pl.col("AN").is_in(base_ans))
            pos = df.filter(~pl.col("floored")).select(["AN", "ID"])
            if (
                pos.height != base_df.height
                or pos.join(
                    base_df.select(["AN", "ID"]), on=["AN", "ID"], how="anti"
                ).height
            ):
                raise ValueError(
                    f"{g}/floor_1bp: positive-length rows differ from the main run"
                )
            df = df.with_row_index("_order")

        else:
            df = gene_gene_only(base_df)
        df = df.sort("_order").drop("_order")

        counts = polarity_counts(df)
        lost_ans = base_ans - set(df["AN"].unique())
        per_genome = df.group_by("AN").agg(
            (pl.col("polarity_3bin") == "conv").any().alias("has_conv"),
            (pl.col("polarity_3bin") == "div").any().alias("has_div"),
        )
        no_conv = per_genome.filter(~pl.col("has_conv")).height
        no_div = per_genome.filter(~pl.col("has_div")).height

        delta = {k: counts[k] - base_counts[k] for k in counts}
        n_floored = df["floored"].sum() if "floored" in df.columns else 0
        print(
            f"  {run:15s} n {df.height:>9,} IGRs ({df.height / base_df.height:6.1%} of main) | "
            f"change same {delta['same']:+,}, conv {delta['conv']:+,}, div {delta['div']:+,} | "
            f"floored {n_floored:,} | "
            f"genomes without conv {no_conv:,}, without div {no_div:,}{extra}"
        )

        if lost_ans:
            msg = f"{g}/{run}: {len(lost_ans):,} genome(s) lost every IGR, e.g. {sorted(lost_ans)[:5]}"
            print(f"    WARNING {msg}")
        for k, n in counts.items():
            if n == 0:
                msg = f"{g}/{run}: no '{k}' IGRs left; the polarity contrast cannot be fitted"
                print(f"    WARNING {msg}")
        if run == "gene_gene_only" and df["flanking_types"].n_unique() != 1:
            raise ValueError(f"{g}/{run}: expected a single flanking_types level")

        write_refit(g, run, df)
    print()

In [ ]:
for g in groups:
    for run in REFITS:
        d = BASE / g / REFIT_DIR_NAME / run
        f = d / "brms_input.tsv"
        print(f"{str(f.relative_to(BASE)):55s} {f.stat().st_size / 1024 ** 2:8.2f} MB")

## Polarity check


In [ ]:
pl.Config.set_tbl_rows(50)
for g in groups:
    gdir = BASE / g
    igr = pl.read_csv(gdir / f"{g}_summary_igr.tsv", separator="\t")

    # value counts for Polarity
    print(f"\nPolarity counts for {g}:")
    print(f"{g:16s} {igr['Polarity'].value_counts().sort('count', descending=True)}")
    print()